## clean the heating index and build the target

updates

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [3]:
raw = Path("../data/raw")
processed = Path("../data/processed")
output = Path("../output")
processed.mkdir(parents=True, exist_ok=True)
output.mkdir(parents=True, exist_ok=True)

## Decisions used in this notebook

| Decision | Value | Why |
|---|---|---|
| First year of the target | 2023 | Observations, time span and type. The heating index file has one row per building and year: 249,398 rows for 21,206 buildings, mainly 2011–2026. We keep one value per building, because yearly values of the same building are so similar that the model could copy the answer across training and test data. We use each building’s latest 3-year average ending in 2023 or later. Older averages would describe buildings before renovations or heating changes, and values have fallen overall (median about 490 in 2018, about 380 in 2025). Newer ones only would cost about 400 of our few houses. After cleaning and merging: 15,277 buildings, 1,415 of them houses. Cross-section: each building appears once.|
| Lowest plausible 3-year average | 50 MJ/m² per year | even very efficient buildings (Minergie-P) are above this; lower values point to a wrong floor area |
| Highest plausible 3-year average | 1,500 MJ/m² per year | about 3× the threshold; higher values point to data errors |

In [4]:
FIRST_YEAR = 2023
MIN_VALUE = 50
MAX_VALUE = 1500

## Cleaning the heating index

We only want the distinct buildings, hence the SCANE_INDICE_MOYENNES_3_ANS file contains buildings multiple times. 

In [7]:
idc = pd.read_csv(raw / "SCANE_INDICE_MOYENNES_3_ANS.csv", sep=";", encoding="utf-8-sig", low_memory=False)

print("Rows:", len(idc))
print("Buildings (unique EGID):", idc["EGID"].nunique())
print("Years:", idc["ANNEE"].min(), "to", idc["ANNEE"].max())

Rows: 249398
Buildings (unique EGID): 21206
Years: 0 to 2026


## Step 2 – Remove duplicates and year 0

Some building-year combinations appear more than once. We first check that the copies carry the same value;
if they do, we can safely keep one of them.

In [8]:
dup_mask = idc.duplicated(["EGID", "ANNEE"], keep=False)
dups = idc[dup_mask]
same_value = dups.groupby(["EGID", "ANNEE"])["INDICE"].nunique().eq(1).all()
print("Rows that are duplicated:", dup_mask.sum())
print("Do all copies have the same value?", same_value)

idc = idc.drop_duplicates(["EGID", "ANNEE"], keep="last")
idc = idc[idc["ANNEE"] != 0]
print("Rows after cleaning:", len(idc))

Rows that are duplicated: 1047
Do all copies have the same value? True
Rows after cleaning: 248859


In [9]:
has_avg = idc[(idc["ANNEE"] >= FIRST_YEAR) & idc["INDICE_MOY3"].notna()]
latest = has_avg.sort_values("ANNEE").drop_duplicates("EGID", keep="last")

target = latest[["EGID", "ANNEE", "INDICE_MOY3", "SRE", "DESTINATION"]].rename(columns={
    "ANNEE": "ref_year", "INDICE_MOY3": "target", "SRE": "sre", "DESTINATION": "destination"})
print("Buildings with a 3-year average from", FIRST_YEAR, ":", len(target))

Buildings with a 3-year average from 2023 : 16587
